In [1]:
import os
from pathlib import Path
import json
from datetime import datetime
import asyncio
import nest_asyncio
from concurrent.futures import ThreadPoolExecutor
from deepeval import evaluate
from deepeval.evaluate import CacheConfig
from deepeval.metrics import GEval, AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.dataset import EvaluationDataset

from tqdm.notebook import tqdm
from dotenv import load_dotenv
load_dotenv()

from agent import MyAgent

In [2]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
timestamp

'20250717_1149'

In [3]:
nest_asyncio.apply()
executor = ThreadPoolExecutor(max_workers=5)
semaphore = asyncio.Semaphore(3)

In [4]:
QS_PATH = Path("generated_QUERIES_Polish_16_07.json")

In [5]:
with open(QS_PATH, "r", encoding="utf-8") as f:
    qs_ans = json.load(f)

In [6]:
def get_retrival_context(input: str) -> str:
    q = [q for q in qs_ans if q["question"]==input][0]
    filename = q['filename']

    with open(f"./data/events/{filename}",'r', encoding='utf-8') as f:
        content = f.read()

    return content



In [7]:
retrival_context_dict = {q["question"]:get_retrival_context(q["question"]) for q in qs_ans}

In [8]:
async def generate_test_case(question: str, expected_answer: str) -> list[LLMTestCase]:
    async with semaphore:
        loop = asyncio.get_event_loop()
        answer = await loop.run_in_executor(
            executor,
            MyAgent().step,
            question
        )
        test_case = LLMTestCase(
            input=question,
            actual_output=answer,
            expected_output=expected_answer,
            retrieval_context=[retrival_context_dict[question]]
        )
        return test_case

async def generate_dataset(qs_ans: list[dict]) -> list[LLMTestCase]:
    tasks = []
    for q in qs_ans:
        question = q["question"]
        expected_answer = q["answer"]
        tasks.append(generate_test_case(question, expected_answer))
    all_test_cases = []
    with tqdm(total=len(tasks), desc="Generating test cases") as pbar:
        for coro in asyncio.as_completed(tasks):
            try:
                result = await coro
                all_test_cases.append(result)
                pbar.update(1)
            except Exception as e:
                print(f"Error processing task: {e}")
                pbar.update(1)
                continue
    return all_test_cases

async def main():
    qs = qs_ans
    all_test_cases = await generate_dataset(qs)
    return all_test_cases

In [9]:
test_cases = await main()

Generating test cases:   0%|          | 0/50 [00:00<?, ?it/s]


----- Step 1 -----


----- Step 1 -----


----- Step 1 -----

Action: parse_user_query

Think: Użytkownik szuka spotkania networkingowego dla przedsiębiorców w Warszawie w 2025 roku. Najpierw należy wyodrębnić kluczowe informacje z zapytania, aby móc skutecznie wyszukać odpowiednie wydarzenia.

Result: Think: Użytkownik szuka spotkania networkingowego skierowanego do przedsiębiorców w Warszawie w roku 2025. W zapytaniu jest jasno określone miasto (Warszawa) oraz rok (2025). Nie podano konkretnego miesiąca ani dnia, więc przyjmuję pełny zakres roku 2025 jako czas poszukiwania wydarzeń.
Parsed user query:
Timeframe: 2025-01-01-2025-12-31
City: Warszawa
Location: None
Refined User Query: spotkanie networkingowe dla przedsiębiorców w Warszawie w 2025 roku
The user's requirements have been successfully extracted. Your required next action is `search_event_pages` to find a list of potential events.


----- Step 2 -----

Action: search_event_pages

Think: Teraz wyszukam wydarzenia odpowiadaj

In [10]:
dataset = EvaluationDataset()
# dataset.add_test_cases_from_json_file(
#     file_path="./deval_datasets/20250717_101820.json",
#     input_key_name="input",
#     actual_output_key_name="actual_output",
#     expected_output_key_name="expected_output"

# )

In [11]:
for test_case in test_cases:
    dataset.add_test_case(test_case)

In [12]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='spotkanie networkingowe dla przedsiębiorców w Warszawie w 2025 roku', actual_output="After reviewing the options, I found an event that's a great match for your request!\n\n\n\tTitle:  Spotkanie networkingowe\n\n\tDate:  Wednesday, July 23, 2025 at 05:00 PM\n\n\tEnd Date: Wednesday, July 23, 2025 at 08:00 PM\n\n\tPrice: Wydarzenie bezpłatne, wymaga rejestracji przez formularz.\n\n\tLocation:  Wawerska Strefa Przedsiębiorczości, Żegańska 1a, sala 49\n\n\tBrief Summary: Spotkanie networkingowe dla przedsiębiorców i osób zainteresowanych biznesem z Warszawy, z wykładem o zastosowaniu sztucznej inteligencji i możliwością networkingu.\n\n\tMore details can be found here: https://wawer.um.warszawa.pl/-/spotkanie-networkingowe\n\nWhy this is a good match:\nWszystkie kryteria użytkownika zostały spełnione: temat, lokalizacja, data i typ wydarzenia. Wydarzenie jest idealnym dopasowaniem do wymagań użytkownika.", expected_output='Title: Spotkanie 

In [13]:
dataset.save_as(file_type="json", directory="deval_datasets", include_test_cases=True, file_name=f"queries_polish_16_07_{timestamp}")

Evaluation dataset saved at deval_datasets/queries_polish_16_07_20250717_1149.json!


'deval_datasets/queries_polish_16_07_20250717_1149.json'

In [14]:
custom_metric = GEval(
    name="Factual Correctness",
    model="gpt-4.1-mini",
    # criteria="Determine if the 'actual_output' is correct based on the 'expected_output'.",
    # criteria="Does the 'actual_output' contain the same information as the 'expected_output'?",
    # criteria="Determine whether the 'actual_output' is factually correct based on the 'expected_output'.",
    criteria="Determine whether the 'actual_output' event details (location, title,date) are factually correct based on the the 'expected_output' event details. Slight deviations in location name are acceptable.",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    threshold=0.5)

answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4.1-mini",
    include_reason=True
)

faithfulness_metric = FaithfulnessMetric(
    threshold=0.6,
    model="gpt-4.1-mini",
    include_reason=True
)

In [15]:
evaluate(
    test_cases=dataset.test_cases,
    metrics=[custom_metric,
             answer_relevancy_metric,
             faithfulness_metric],
    cache_config=CacheConfig(use_cache=True))

✨ You're running DeepEval's latest Factual Correctness [GEval] Metric! (using gpt-4.1-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=True)...

Output()

ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in 
organization org-O4F9bo5U8nyc6XBWWtESr2Yg on tokens per min (TPM): Limit 200000, Used 200000, Requested 665. Please
try again in 199ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 
'param': None, 'code': 'rate_limit_exceeded'}} Retrying: 1 time(s)...

...

ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in 
organization org-O4F9bo5U8nyc6XBWWtESr2Yg on tokens per min (TPM): Limit 200000, Used 200000, Requested 730. Please
try again in 219ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 
'param': None, 'code': 'rate_limit_exceeded'}} Retrying: 3 time(s)...

ERROR:root:OpenAI Error: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4.1-mini in 
organization org-O4F9bo5U8nyc6XBWWtESr2Yg on tokens per min (TPM): Limit 200000, Used 199685, Requested 820. Please
try again in 151ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 
'param': None, 'code': 'rate_limit_exceeded'}} Retrying: 3 time(s)...



Metrics Summary

  - ✅ Factual Correctness [GEval] (score: 0.85, threshold: 0.5, strict: False, evaluation model: gpt-4.1-mini, reason: The title matches exactly, and the location is factually correct with acceptable formatting differences. The date in the actual output specifies a single occurrence on July 3, 2025, while the expected output includes recurring dates on subsequent Thursdays in July and August, indicating partial but not full alignment on the date field. Overall, the three fields represent the same event, but the date coverage is less complete in the actual output., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gpt-4.1-mini, reason: The score is 1.00 because the response fully addresses the query about summer 2025 art activities for children in Warsaw without any irrelevant information. It is concise and on-topic, making it perfectly relevant., error: None)
  - ✅ Faithfulness (score: 1.0, threshold: 0.6, strict: False

✓ Tests finished 🎉! Run 'deepeval view' to analyze, debug, and save evaluation results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_2', success=True, metrics_data=[MetricData(name='Factual Correctness [GEval]', threshold=0.5, success=True, score=0.85, reason='The title matches exactly, and the location is factually correct with acceptable formatting differences. The date in the actual output specifies a single occurrence on July 3, 2025, while the expected output includes recurring dates on subsequent Thursdays in July and August, indicating partial but not full alignment on the date field. Overall, the three fields represent the same event, but the date coverage is less complete in the actual output.', strict_mode=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0006668, verbose_logs='Criteria:\nDetermine whether the \'actual_output\' event details (location, title,date) are factually correct based on the the \'expected_output\' event details. Slight deviations in location name are acceptable. \n \nEvaluation Steps:\n[\n    "Compare the

In [16]:
def get_question(input: str) -> None:
    q = [q for q in qs_ans if q["question"]==input][0]
    print(f"Question: {q['question']}")
    print(f"Answer: {q['answer']}")
    print(f"filename: {q['filename']}")

# get_question("Jakie są wydarzenia związane z indywidualnym zwiedzaniem wystaw historycznych w Warszawie w czerwcu 2025?")

In [17]:
os.rename("./.deepeval/.deepeval-cache.json", f"./.deepeval/.deepeval-cache_{timestamp}.json")
os.rename("./.deepeval/.deepeval_telemetry.txt", f"./.deepeval/.deepeval_telemetry_{timestamp}.txt")